### Dynamic Models

In [24]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.messages import HumanMessage, AIMessage
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from typing import Callable
from pprint import pprint

Globals

In [25]:
large_model = init_chat_model(f"bedrock:us.anthropic.claude-opus-4-6-v1")
standard_model = init_chat_model(f"bedrock:us.anthropic.claude-haiku-4-5-20251001-v1:0")

Model Middle Ware

In [26]:
@wrap_model_call
def state_based_model(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    
    message_count = len(request.messages)  

    if message_count > 10:
        model = large_model
    else:
        model = standard_model

    request = request.override(model=model)  
    return handler(request)

Agent

In [27]:
agent = create_agent(
    standard_model,
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

Dynamic Model Invocation for Standard Model

In [ ]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

for m in response['messages']:
    m.pretty_print()

I appreciate the question, but I should clarify - I'm Claude, an AI assistant. I'm not actually a physical person in your office, so I can't water plants or perform other physical tasks.

However, if you're looking for help with office plant care, I'd be happy to:
- Help you create a watering schedule
- Suggest plant care tips
- Remind you to check if the soil is dry
- Recommend low-maintenance office plants

Is there something specific about office plant care I can help with?


In [29]:
print(response["messages"][-1].response_metadata["model_name"])

us.anthropic.claude-haiku-4-5-20251001-v1:0


Dynamic Mode Invocation for Large Model

In [30]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

Hmm, good question. Based on how fast it's been growing, I'd say we'll probably need to repot it in the next couple of months. Once you start seeing roots poking out of the drainage holes at the bottom, that's usually the sign it's time. I can keep an eye on it and let you know when it looks like it's getting root-bound. We might want to go up one pot size — maybe from the current 6-inch to an 8-inch pot. Want me to look into ordering one ahead of time so we're ready?


In [31]:
print(response["messages"][-1].response_metadata["model_name"])

us.anthropic.claude-opus-4-6-v1
